<a href="https://colab.research.google.com/github/BahruzHuseynov/Portfolio/blob/main/Deep_Learning_Projects/CNN_Architecture/PyTorch/CNN2_AlexNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AlexNet

<img src = "https://www.oreilly.com/api/v2/epubs/9781789956177/files/assets/ec08175c-5282-4be2-b6e7-6f2d99272166.png">

<img src = "https://i0.wp.com/syncedreview.com/wp-content/uploads/2017/05/13.png?resize=330%2C230&ssl=1">

## Import libraries

In [1]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from torch import nn
import torch.nn.functional as F
from torch import optim

## Dataset creation

In [2]:
!pip install -q kaggle

In [3]:
from google.colab import files
files.upload()

Saving kaggle (1).json to kaggle (1).json


{'kaggle (1).json': b'{"username":"hbahruz","key":"4f925bc35f1b4f8677fc6f8061cceb06"}'}

In [4]:
!kaggle datasets download -d mdismielhossenabir/crab-species-datasets

Dataset URL: https://www.kaggle.com/datasets/mdismielhossenabir/crab-species-datasets
License(s): MIT
  0% 0.00/3.58M [00:00<?, ?B/s]
100% 3.58M/3.58M [00:00<00:00, 75.0MB/s]


In [5]:
import zipfile

crab_species_zip = 'crab-species-datasets.zip'

def extract_zip(file_path, extract_to='.'):
    with zipfile.ZipFile(file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

extract_zip(crab_species_zip, 'crab_species_dataset')

In [6]:
transform = transforms.Compose([
    transforms.Resize((227, 227)),  # Resize images into AlexNet input
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [7]:
main_dir = "crab_species_dataset"
train_dir = main_dir + "/train/train"
test_dir = main_dir + "/test/test"

train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

## Model Building

In [8]:
class AlexNet(nn.Module):
    def __init__(self, num_classes, in_channels = 3):
        super(AlexNet, self).__init__()
        self.num_classes = num_classes
        self.layers = ["conv","maxpool", "conv", "maxpool", "conv", "conv", "conv", "maxpool"]
        self.out_channels = [96, 96, 256, 256, 384, 384, 256, 256]
        self.strides = [4, 2, 1, 2, 1, 1, 1, 2]
        self.pads = [0, 0, 2, 0, 1, 1, 1, 0]
        self.kernels = [11, 3, 5, 3, 3, 3, 3, 3]

        self.seq = nn.Sequential()

        layer = 0
        while layer < len(self.layers):
            if self.layers[layer] == "conv":
                input_channel = in_channels
                if layer != 0:
                    input_channel = self.out_channels[layer-1]

                new_layer = nn.Conv2d(in_channels = input_channel,  out_channels = self.out_channels[layer],
                                      kernel_size = self.kernels[layer], stride = self.strides[layer], padding = self.pads[layer])

                self.seq.add_module(f'layer-{layer}', new_layer)
                self.seq.add_module(f'layer-{layer}-RELU', nn.ReLU())

            if self.layers[layer] == "maxpool":
                new_layer = nn.Conv2d(in_channels = self.out_channels[layer-1], out_channels = self.out_channels[layer],
                                      kernel_size = self.kernels[layer], stride = self.strides[layer], padding = self.pads[layer])

                self.seq.add_module(f'layer-{layer}', new_layer)

            layer += 1

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*6*6, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, self.num_classes),
            # nn.Softmax()
        )

    def forward(self, x):
        x = self.seq(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

In [9]:
model = AlexNet(6)
print(model)

AlexNet(
  (seq): Sequential(
    (layer-0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4))
    (layer-0-RELU): ReLU()
    (layer-1): Conv2d(96, 96, kernel_size=(3, 3), stride=(2, 2))
    (layer-2): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (layer-2-RELU): ReLU()
    (layer-3): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2))
    (layer-4): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (layer-4-RELU): ReLU()
    (layer-5): Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (layer-5-RELU): ReLU()
    (layer-6): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (layer-6-RELU): ReLU()
    (layer-7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2))
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=4096, out_features

## Training

In [10]:
f_loss = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum = 0.9)

In [11]:
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for batch_idx, (samples, targets) in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model(samples)
        loss = f_loss(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    print("Epoch:", epoch + 1, "-->", train_loss)

Epoch: 1 --> 73.8121052980423
Epoch: 2 --> 74.16102397441864
Epoch: 3 --> 74.69408071041107
Epoch: 4 --> 77.33028018474579
Epoch: 5 --> 75.52035319805145


## Testing

In [12]:
from sklearn.metrics import classification_report

In [13]:
model.eval()
preds = []
actual = []
test_loss = 0.0
with torch.no_grad():
    for samples, targets in test_loader:
        outputs = model(samples)
        _, predicted = torch.max(outputs.data, 1)
        preds.extend(predicted.numpy())
        actual.extend(targets.numpy())



In [14]:
print(classification_report(actual, preds))

              precision    recall  f1-score   support

           0       0.17      1.00      0.29         5
           1       0.00      0.00      0.00         5
           2       0.00      0.00      0.00         5
           3       0.00      0.00      0.00         5
           4       0.00      0.00      0.00         5
           5       0.00      0.00      0.00         5

    accuracy                           0.17        30
   macro avg       0.03      0.17      0.05        30
weighted avg       0.03      0.17      0.05        30



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
